<a href="https://colab.research.google.com/github/mitalidaduria/nlp-payments-lab/blob/main/FinSentimentAnalyser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Hugging Face Transformers, Datasets, and Data Manipulation libraries
!pip install -q transformers datasets pandas scikit-learn torch

In [2]:
import pandas as pd
import torch
from transformers import pipeline

class FinSentimentAnalyser:
    def __init__(self):
        print("Loading models into GPU/CPU memory...")
        # General BERT model (DistilBERT trained on SST-2 general sentiment)
        self.general_pipeline = pipeline(
            "text-classification",
            model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
        )

        # Financial Domain Model (ProsusAI/finbert)
        self.finbert_pipeline = pipeline(
            "text-classification",
            model="ProsusAI/finbert"
        )
        print("Models loaded successfully!")

    def compare_models(self, test_phrases: list) -> pd.DataFrame:
        results = []

        for phrase in test_phrases:
            # General Model Prediction
            gen_out = self.general_pipeline(phrase)[0]
            gen_label = gen_out["label"].upper()
            gen_score = round(gen_out["score"], 4)

            # FinBERT Prediction
            fin_out = self.finbert_pipeline(phrase)[0]
            fin_label = fin_out["label"].upper()
            fin_score = round(fin_out["score"], 4)

            # Direct sentiment agreement flag
            agree_flag = (gen_label == fin_label)

            results.append({
                "Financial Text": phrase,
                "General BERT Label": gen_label,
                "General Conf.": gen_score,
                "FinBERT Label": fin_label,
                "FinBERT Conf.": fin_score,
                "Agreement": agree_flag
            })

        return pd.DataFrame(results)

    def analyse_product_feedback(self, df: pd.DataFrame) -> dict:
        total = len(df)
        agreements = int(df["Agreement"].sum())
        disagreements = total - agreements

        positives = (df["FinBERT Label"] == "POSITIVE").sum()
        negatives = (df["FinBERT Label"] == "NEGATIVE").sum()
        nps_proxy = round(((positives - negatives) / total) * 100, 2) if total > 0 else 0.0

        return {
            "Total Evaluated": total,
            "Agreement Count": agreements,
            "Divergence Count": disagreements,
            "Agreement Rate (%)": round((agreements / total) * 100, 2),
            "NPS Proxy Score": nps_proxy
        }

In [3]:
# Initialize Financial Analyser
analyser = FinSentimentAnalyser()

# Test cases highlighting domain-specific financial semantics
fintech_test_cases = [
    "Profit warning issued following lower quarterly revenue",
    "Payment gateway experiencing technical issues during checkout",
    "Dispute resolved successfully by customer support",
    "Company announced restructuring plan to cut operational expenses",
    "Operating margins expanded by 150 basis points",
    "Regulator launched investigation into compliance violations",
    "Liquidity position remains strong despite market headwinds"
]

# Run model comparison
df_results = analyser.compare_models(fintech_test_cases)

# Display tabular output
display(df_results)

# Display aggregate breakdown
metrics = analyser.analyse_product_feedback(df_results)
print("\n--- AGGREGATE FINANCIAL METRICS ---")
for key, value in metrics.items():
    print(f"{key}: {value}")

Loading models into GPU/CPU memory...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Models loaded successfully!


,Financial Text,General BERT Label,General Conf.,FinBERT Label,FinBERT Conf.,Agreement
0,Profit warning issued following lower quarterl...,NEGATIVE,0.9988,NEGATIVE,0.9712,True
1,Payment gateway experiencing technical issues ...,NEGATIVE,0.9939,NEGATIVE,0.9651,True
2,Dispute resolved successfully by customer support,POSITIVE,0.9950,NEUTRAL,0.8036,False
3,Company announced restructuring plan to cut op...,NEGATIVE,0.9976,NEGATIVE,0.9630,True
4,Operating margins expanded by 150 basis points,POSITIVE,0.9898,POSITIVE,0.9560,True
5,Regulator launched investigation into complian...,NEGATIVE,0.8388,NEGATIVE,0.6847,True
6,Liquidity position remains strong despite mark...,POSITIVE,0.9983,POSITIVE,0.9507,True



--- AGGREGATE FINANCIAL METRICS ---
Total Evaluated: 7
Agreement Count: 6
Divergence Count: 1
Agreement Rate (%): 85.71
NPS Proxy Score: -28.57


### Key Insights from Financial NLP Comparison:
1. **Domain-Specific Phrases:** Statements like *"Profit warning issued"* contain positive words (*"profit"*), leading general models to misclassify them as `POSITIVE`. FinBERT understands that a "profit warning" is a negative market indicator.
2. **Contextual Market Terminology:** Operational events (e.g., *"cut operational expenses"* or *"restructuring"*) have distinct financial implications that standard language models often flag as purely negative, whereas domain models weigh them against long-term corporate strategy.